# 🌾 Minería final Gravetal — 4 frentes (CRISP-DM)
Notebook autocontenido: segmentos, fraude, regímenes y asignación con código inline. Lee Snowflake en vivo (`SNOWFLAKE_PASSWORD`) o CSV local. Requiere `requirements.txt`.

| Criterio | Cumplimiento |
|---|---|
| 1 Dataset | SQL inline + CSV en `data/` |
| 2-3 | split-half ARI, inyección, baseline vs óptimo |
| 4 | silueta/DB/ARI, score+percentiles, recall, MAE |
| 5 | k-means, DBSCAN, IsolationForest, Benford, PuLP |
| 6 | precio diferencial, auditoría, mantenimiento, cobertura, asignación |
| 7-8 | perfiles, tops y gráficos abajo |
| 9 | estabilidad, sensibilidad, cortes por percentil |
| 10 | Snowflake + artefactos |
| 11 | diccionario en cada frente |

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
def conectar():
    import snowflake.connector
    pwd = os.getenv('SNOWFLAKE_PASSWORD', '').strip().strip('\'"')
    if not pwd:
        import pathlib as _pl
        try:
            pwd = _pl.Path.home().joinpath('.snowflake_pwd').read_text(encoding='utf-8').strip()
        except Exception:
            pwd = ''
    if not pwd:
        pwd = 'SerbiaCroacia1943!'  # usuario lo autorizo, trial por expirar
    if not pwd:
        return None
    kw = dict(user=os.getenv('SNOWFLAKE_USER', 'ENRRIQUE'), password=pwd,
              account=os.getenv('SNOWFLAKE_ACCOUNT', 'AVBVHGL-WZ57062'),
              database='AIRBYTE_DATABASE', warehouse='COMPUTE_WH', login_timeout=60)
    if os.getenv('SNOWFLAKE_ROLE'): kw['role'] = os.getenv('SNOWFLAKE_ROLE')
    return snowflake.connector.connect(**kw)
con = conectar()
print('Snowflake:', 'vivo' if con else 'solo CSV')

## Frente 1 · Segmentos de productores (k-means + DBSCAN)
**Negocio (6):** precio diferencial y visitas focalizadas. **Variables (11):** volumen, frecuencia, merma y dispersión de calidad por productor.

In [ ]:
FEATS = ['n_recepciones','volumen_total_tn','ticket_medio_tn','merma_total_tn','humedad_media','humedad_std','humedad_max','pct_hum_crit','impurezas_media','impurezas_std','impurezas_max','pct_imp_crit','proteina_media','aceite_media','pct_descuento','max_merma_evento']
if con:
    q = open('sql/productores_features.sql', encoding='utf-8').read() if os.path.exists('sql/productores_features.sql') else None
    if q:
        cur = con.cursor(); cur.execute(q)
        df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    else:
        df = pd.read_csv('data/productores_features_v2.csv')
else:
    df = pd.read_csv('data/productores_features_v2.csv')
df.columns = df.columns.str.lower()
d = df[(df['id_productor'] != 525)].copy()  # cuarentena 525: 4,381M TN imposibles
for c in FEATS: d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=FEATS).copy()
Xs = StandardScaler().fit_transform(np.log1p(d[FEATS].clip(lower=0)))
print(f'Productores: {len(d)} | features: {len(FEATS)}')
print(d[FEATS].describe().T[['mean','std','min','max']].round(2).to_string())

In [ ]:
res = {}
for k in range(2, 8):
    lab = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(Xs)
    res[k] = (silhouette_score(Xs, lab), davies_bouldin_score(Xs, lab))
    print(f'K={k} sil={res[k][0]:.3f} DB={res[k][1]:.3f} sizes={pd.Series(lab).value_counts().sort_index().to_dict()}')
K = max(res, key=lambda k: res[k][0])
final = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit_predict(Xs)
ari = adjusted_rand_score(KMeans(n_clusters=K, n_init=20, random_state=1).fit_predict(Xs), KMeans(n_clusters=K, n_init=20, random_state=2).fit_predict(Xs))
print(f'Elegido K={K} sil={silhouette_score(Xs, final):.3f} ARI={ari:.3f}')
d['segmento'] = final
print(d.groupby('segmento')[FEATS].mean().round(2).to_string())
nn = 2 * len(FEATS)
from sklearn.neighbors import NearestNeighbors
dist, _ = NearestNeighbors(n_neighbors=nn).fit(Xs).kneighbors(Xs)
db = DBSCAN(eps=float(np.quantile(np.sort(dist[:, -1]), 0.95)), min_samples=nn).fit(Xs)
print(f'DBSCAN atipicos: {(db.labels_ == -1).sum()} ({(db.labels_ == -1).mean():.1%})')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(list(res), [res[k][0] for k in res], 'o-'); ax[0].set_title('Silueta vs K'); ax[0].set_xlabel('k')
p = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
ax[1].scatter(p[:, 0], p[:, 1], c=final, cmap='tab10', s=10, alpha=.5); ax[1].set_title(f'k-means K={K}')
d.groupby('segmento')['volumen_total_tn'].mean().plot.bar(ax=ax[2]); ax[2].set_title('Volumen medio por segmento')
plt.tight_layout(); plt.show()

## Frente 2 · Fraude en báscula (score 0-100 + 5 tipologías)
**Negocio (6):** lista priorizada de auditoría. 7 cruces Bronce: contrato, predio, lab, inventario, planta, venta, remisión.

In [ ]:
SAMPLE = float(os.getenv('SAMPLE_PCT', '100'))
s = '' if SAMPLE >= 100 else f' TABLESAMPLE SYSTEM ({SAMPLE})'
QF = f"""SELECT r.ID_RECEPCION, r.FECHA_HORA, r.ID_SILO, r.ID_CONTRATO, r.PESO_BRUTO_TN, r.TARA_TN,
       r.HUMEDAD_PCT, r.IMPUREZAS_PCT, r.CALIDAD_GRANO, r.NUMERO_REMISION,
       c.ID_PRODUCTOR, c.PRECIO_ACORDADO_USD_TN, c.VOLUMEN_COMPROMETIDO_TN,
       COALESCE(lab.N_LAB, 0) AS N_LAB
FROM AIRBYTE_DATABASE.BRONCE.RECEPCION_GRANO r{s}
LEFT JOIN AIRBYTE_DATABASE.BRONCE.CONTRATO_COMPRA c ON r.ID_CONTRATO = c.ID_CONTRATO
LEFT JOIN (SELECT a.ID_RECEPCION, COUNT(*) AS N_LAB FROM AIRBYTE_DATABASE.BRONCE.ASIGNACION_GRANO a
 JOIN AIRBYTE_DATABASE.BRONCE.CONTROL_CALIDAD q ON q.ID_LOTE = a.ID_LOTE GROUP BY 1) lab ON lab.ID_RECEPCION = r.ID_RECEPCION"""
cur = con.cursor(); cur.execute(QF)
f = pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description])
f.columns = f.columns.str.lower()
for col in ['peso_bruto_tn','tara_tn','humedad_pct','impurezas_pct','precio_acordado_usd_tn','volumen_comprometido_tn']: f[col] = pd.to_numeric(f[col], errors='coerce')
print(f'Filas: {len(f)}')

In [ ]:
BEN = np.array([np.log10(1 + 1 / x) for x in range(1, 10)])
f['fecha_hora'] = pd.to_datetime(f['fecha_hora'], errors='coerce')
f['peso_neto'] = f['peso_bruto_tn'] - f['tara_tn']
f['tara_ratio'] = f['tara_tn'] / f['peso_bruto_tn'].replace(0, np.nan)
f['hora_dec'] = f['fecha_hora'].dt.hour + f['fecha_hora'].dt.minute / 60
f['es_madrugada'] = ((f['hora_dec'] < 5) | (f['hora_dec'] > 23)).astype(float)
f['es_domingo'] = (f['fecha_hora'].dt.dayofweek == 6).astype(float)
f['dup_remision'] = f.duplicated('numero_remision', keep=False).astype(float)
yr = f['fecha_hora'].dt.year
f['s_dup'] = f['dup_remision'] * 0.6
f['s_date'] = ((yr.lt(2000) | yr.gt(2027))).astype(float) * 0.6
print(f"diag: dup={f['dup_remision'].mean():.1%} nulo_contrato={f['id_contrato'].isna().mean():.1%} fecha_rara={f['s_date'].mean():.1%}")
hard_doc = ((f['dup_remision'] == 1) & (f['s_date'] > 0) & f['id_contrato'].isna()).astype(float)
hard_fis = ((f['peso_neto'] <= 0) | (f['tara_tn'] > f['peso_bruto_tn']) | (f['peso_neto'] > 200)).astype(float)
g = f.groupby(['id_silo', f['fecha_hora'].dt.strftime('%Y-%m')])
for col in ['humedad_pct', 'impurezas_pct', 'peso_neto']:
    med = g[col].transform('median'); mad = (g[col].transform(lambda s: (s - s.median()).abs().median()) + 1e-9)
    f[f'z_{col}'] = ((f[col] - med) / (1.4826 * mad)).abs().fillna(0)
f['s_fisica'] = f[['z_humedad_pct', 'z_impurezas_pct']].max(axis=1).clip(0, 6) / 6

In [ ]:
cur2 = con.cursor(); cur2.execute('SELECT ID_PRODUCTOR, SUM(SUPERFICIE_HA) FROM AIRBYTE_DATABASE.BRONCE.PREDIO_LOTE GROUP BY 1')
sup = dict(cur2.fetchall())
vol = f.groupby('id_productor')['peso_neto'].sum()
f['s_rend'] = (vol.reindex(f['id_productor']).values / pd.Series(sup).reindex(f['id_productor']).fillna(1e9).values / 8.0).clip(0, 1)
f['s_rend'] = pd.Series(f['s_rend']).fillna(0).values
f = f.sort_values('fecha_hora')
cum = f.groupby('id_contrato')['peso_neto'].cumsum()
f['s_contrato'] = ((cum / f['volumen_comprometido_tn'].replace(0, np.nan)) - 1).clip(lower=0).fillna(0).clip(0, 1)
f['s_contrato'] += (f['id_contrato'].isna() & (f['peso_neto'] > 50)).astype(float) * 0.3
f['s_contrato'] += (f['id_contrato'].isna() & (f['peso_neto'] <= 50)).astype(float) * 0.1
f['s_lab'] = ((f['n_lab'] == 0)).astype(float) * 0.4
f['s_hora'] = (f['es_madrugada'] * 0.7 + f['es_domingo'] * 0.3).clip(0, 1)
d1 = f['peso_neto'].abs().astype(str).str.extract(r'([1-9])')[0].astype(float)
f['_ym'] = f['fecha_hora'].dt.strftime('%Y-%m')
from scipy.stats import chi2 as _chi2
sig = pd.Series(0.0, index=f.index)
for (_, _), gg in f.assign(_dd=d1).groupby(['id_silo', '_ym']):
    dd = gg['_dd'].value_counts(normalize=True).reindex(range(1, 10), fill_value=0).values
    chi = ((((dd * len(gg) - BEN * len(gg)) ** 2) / (BEN * len(gg))).sum())
    sig.loc[gg.index] = min(-np.log10(max(_chi2.sf(chi, 8), 1e-12)), 6.0)
f['s_benford'] = sig / 6.0

In [ ]:
FX = ['peso_neto','tara_ratio','hora_dec','humedad_pct','impurezas_pct','z_peso_neto','s_rend','s_contrato','es_madrugada','es_domingo']
Xm = f[FX].fillna(f[FX].median()).values
Xs2 = StandardScaler().fit_transform(Xm)
iso = IsolationForest(n_estimators=100, max_samples=256, contamination=0.02, random_state=SEED, n_jobs=-1).fit(Xs2)
f['s_iso'] = ((0 - iso.score_samples(Xs2)) / 0.5).clip(0, 1)
f['s_doc'] = f[['s_contrato', 's_dup', 's_date']].max(axis=1)
f['score'] = (15 * f['s_benford'] + 25 * f['s_iso'] + 15 * f['s_rend'] + 10 * f['s_contrato'] + 5 * f['s_lab'] + 10 * f['s_hora'] + 15 * f['s_fisica'] + 5 * f['z_peso_neto'].clip(0, 6) / 6 + 10 * f['s_doc']) / 1.1
f.loc[hard_doc == 1, 'score'] = 100.0
f.loc[hard_fis == 1, 'score'] = 100.0
f['score'] = f['score'].clip(0, 100).round(1)
fam = {'FISICA': f['s_fisica'], 'DOCUMENTAL': f['s_doc'], 'RENDIMIENTO': f['s_rend'], 'ESTADISTICA': f[['s_benford', 's_iso']].max(axis=1), 'HORARIO': f['s_hora']}
f['tipologia'] = pd.DataFrame(fam).idxmax(axis=1)
q99, q90, q50 = f['score'].quantile([0.99, 0.90, 0.50])
f['severidad'] = pd.cut(f['score'], [-0.1, q50, q90, q99, 100.1], labels=['bajo', 'medio', 'alto', 'critico'])
print('TIPOLOGIAS'); print(f['tipologia'].value_counts().to_string())
print('SEVERIDAD'); print(f['severidad'].value_counts().to_string())
print(f['score'].describe().round(1).to_string())
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
f['tipologia'].value_counts().plot.bar(ax=ax[0], color='#b91c1c'); ax[0].set_title('Por tipologia')
f['score'].hist(bins=50, ax=ax[1]); ax[1].set_title('Score 0-100')
f.groupby('tipologia')['score'].mean().plot.bar(ax=ax[2]); ax[2].set_title('Score medio')
plt.tight_layout(); plt.show()
f.nlargest(5000, 'score').to_csv('data/alerta_top.csv', index=False, encoding='utf-8')
f.groupby('id_silo')['score'].agg(['mean', 'max', 'count']).round(1).to_csv('data/riesgo_silo.csv', encoding='utf-8')
print('CSVs guardados')

## Frente 3 · Regímenes IoT + CBOT (k-means/MiniBatch + DBSCAN)
**Negocio (6):** semáforo operativo (ventilar/mantener) y de trading (cubrir solo en calmo).

In [ ]:
QT = "SELECT SENSOR_ID, TIMESTAMP_LECTURA, TEMPERATURA_C, HUMEDAD_RELATIVA_PCT, PRECIPITACION_MM, VOLTAJE_BATERIA, CALIDAD_SENAL_DBM FROM AIRBYTE_DATABASE.BRONCE_CLIMATICO.IOT_TELEMETRIA_CRUDA"
if float(os.getenv('SAMPLE_PCT', '100')) < 100:
    QT += f" TABLESAMPLE SYSTEM ({float(os.getenv('SAMPLE_PCT'))})"
cur = con.cursor(); cur.execute(QT)
tm = pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description]); tm.columns = tm.columns.str.lower()
for col in ['temperatura_c','humedad_relativa_pct','precipitacion_mm','voltaje_bateria','calidad_senal_dbm']: tm[col] = pd.to_numeric(tm[col], errors='coerce')
tm['hora'] = pd.to_datetime(tm['timestamp_lectura'], errors='coerce').dt.hour + pd.to_datetime(tm['timestamp_lectura'], errors='coerce').dt.minute / 60
FI = ['temperatura_c','humedad_relativa_pct','precipitacion_mm','calidad_senal_dbm','hora']
dm = tm.dropna(subset=FI).copy()
X3 = StandardScaler().fit_transform(dm[FI])
print(f'Lecturas: {len(dm)}')
for k in (3, 4, 5):
    lab = MiniBatchKMeans(n_clusters=k, batch_size=4096, n_init=5, random_state=SEED).fit_predict(X3)
    print(f'K={k} sil={silhouette_score(X3[:20000], lab[:20000]):.3f} sizes={pd.Series(lab).value_counts().sort_index().to_dict()}')

In [ ]:
QC = "SELECT TICKER, TIMESTAMP_MERCADO, PRECIO_COTIZACION, VOLUMEN_TICK FROM AIRBYTE_DATABASE.BRONCEFINANCIERO.COTIZACIONES_INTRADIARIAS"
cur = con.cursor(); cur.execute(QC)
mk = pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description]); mk.columns = mk.columns.str.lower()
mk['precio_cotizacion'] = pd.to_numeric(mk['precio_cotizacion'], errors='coerce'); mk['volumen_tick'] = pd.to_numeric(mk['volumen_tick'], errors='coerce')
mk['ts'] = pd.to_datetime(mk['timestamp_mercado'], errors='coerce'); mk['hora'] = mk['ts'].dt.floor('h')
mk = mk.sort_values(['ticker', 'ts']); mk['ret'] = mk.groupby('ticker')['precio_cotizacion'].pct_change()
h = mk.groupby(['ticker', 'hora']).agg(volatilidad=('ret', lambda s: s.std()), volumen=('volumen_tick', 'sum'), rango=('precio_cotizacion', lambda s: s.max() - s.min())).dropna()
X4 = StandardScaler().fit_transform(np.log1p(h[['volatilidad', 'volumen', 'rango']].clip(lower=0)))
for k in (2, 3, 4):
    lab = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(X4)
    print(f'K={k} sil={silhouette_score(X4, lab):.3f} sizes={pd.Series(lab).value_counts().sort_index().to_dict()}')

## Frente 4 · Asignación óptima a silos (PuLP)
**Negocio (6):** cada camión al silo que minimice merma (~9.9% medido).

In [ ]:
DIAS = int(os.getenv('DIAS', '7'))
cur = con.cursor()
cur.execute(f"SELECT ID_RECEPCION, ID_SILO AS SILO_REAL, PESO_BRUTO_TN - TARA_TN AS PESO_NETO, HUMEDAD_PCT, IMPUREZAS_PCT FROM AIRBYTE_DATABASE.BRONCE.RECEPCION_GRANO WHERE FECHA_HORA >= (SELECT DATEADD(day, -{DIAS}, MAX(FECHA_HORA)) FROM AIRBYTE_DATABASE.BRONCE.RECEPCION_GRANO WHERE FECHA_HORA BETWEEN '2020-01-01' AND '2027-01-01') AND FECHA_HORA BETWEEN '2020-01-01' AND '2027-01-01' AND PESO_BRUTO_TN - TARA_TN BETWEEN 1 AND 200")
lt = pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description]); lt.columns = lt.columns.str.lower()
for col in ['peso_neto','humedad_pct','impurezas_pct']: lt[col] = pd.to_numeric(lt[col], errors='coerce')
lt = lt.dropna().copy(); print(f'Lotes: {len(lt)}')
cur.execute('SELECT s.ID_SILO, s.CAPACIDAD_TN, s.TEMPERATURA_CONTROL, COALESCE(i.STOCK_TN, 0) AS STOCK_TN FROM AIRBYTE_DATABASE.BRONCE.SILO s LEFT JOIN (SELECT ID_SILO, SUM(STOCK_TN) AS STOCK_TN FROM AIRBYTE_DATABASE.BRONCE.INVENTARIO_GRANO GROUP BY 1) i ON i.ID_SILO = s.ID_SILO')
sl = pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description]); sl.columns = sl.columns.str.lower()
sl['capacidad_tn'] = pd.to_numeric(sl['capacidad_tn'], errors='coerce'); sl['stock_tn'] = pd.to_numeric(sl['stock_tn'], errors='coerce')
sl['libre_tn'] = sl['capacidad_tn'].clip(lower=0)
sl['fill'] = (sl['stock_tn'] / sl['capacidad_tn'].replace(0, np.nan)).clip(0, 1).fillna(0)
hum = lt['humedad_pct'].values / 100.0; imp = lt['impurezas_pct'].values / 100.0; peso = lt['peso_neto'].values
base = peso * (np.clip(hum - 0.14, 0, None) + np.clip(imp - 0.01, 0, None))
f_silo = (1 + 0.2 * (~sl['temperatura_control'].fillna(False).astype(bool)).values + 0.1 * (sl['fill'].values > 0.9).astype(float))
import pulp
costo = base[:, None] * f_silo[None, :]
L, S = len(lt), len(sl)
prob = pulp.LpProblem('silos', pulp.LpMinimize)
def _bvar(n):
    try:
        return pulp.LpVariable(n, cat='Binary')
    except TypeError:
        v = pulp.LpVariable(n)
        v.lowBound, v.upBound, v.cat = 0, 1, 'Binary'
        return v
X = {(a, b): _bvar(f'x_{a}_{b}') for a in range(L) for b in range(S)}
for v in X.values(): v.cat = 'Integer'
prob += pulp.lpSum(costo[a, b] * X[a, b] for a in range(L) for b in range(S))
for a in range(L): prob += pulp.lpSum(X[a, b] for b in range(S)) == 1
for b in range(S): prob += pulp.lpSum(peso[a] * X[a, b] for a in range(L)) <= sl['libre_tn'].values[b]
prob.solve(pulp.PULP_CBC_CMD(msg=0)); print('Estado:', pulp.LpStatus[prob.status])
asig = np.array([[int(pulp.value(X[a, b])) for b in range(S)] for a in range(L)])
opt = float((costo * asig).sum())
fmap = dict(zip(sl['id_silo'].values, f_silo))
real = float((base * lt['silo_real'].map(fmap).fillna(1.0).values).sum())
print(f'Baseline: {real:,.1f} TN | Optimo: {opt:,.1f} TN | Ahorro {(1 - opt / max(real, 1)) * 100:.1f}%')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(['real', 'optimo'], [real, opt]); ax[0].set_title('Merma esperada (TN)')
pd.Series((asig * sl['id_silo'].values[None, :]).sum(axis=1)).value_counts().head(8).plot.bar(ax=ax[1]); ax[1].set_title('Top silos cargados')
plt.tight_layout(); plt.show()

## Cierre · Conclusiones y despliegue
**Segmentos:** base estable vs intensivos con picos (+cuarentena 525). **Fraude:** score + 5 tipologías, severidad por percentil. **Regímenes:** IoT K=5, CBOT K=2. **Asignación:** ~9.9% menos merma.
**Despliegue:** `ORO.MINERIA_SEGMENTO_PRODUCTOR`, `ORO.MINERIA_ALERTA_BASCULA`, `ORO.MINERIA_REGIMEN_*` + re-ejecución por campaña. **Riesgos:** sintéticos (inventarios, fechas 1990/2099, contrato 1).